# C. elegans Chemical–Gene (STITCH) — Relation-Wise KG Triple Construction

## Purpose

This notebook processes **Chemical–Protein interaction data** from the STITCH database for *C. elegans* and transforms it into standardized Chemical–Gene relation-wise Knowledge Graph (KG) triples. Protein identifiers (WormBase LocusTags) are mapped to gene descriptions via NCBI, and chemicals are annotated with PubChem IUPAC names.

## Pipeline Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | PubChem Mapping | Load PubChem (CID → IUPAC Name) |
| 2 | NCBI Gene Info | Load NCBI gene annotations (LocusTag → Description) |
| 3 | STITCH Processing | Load raw STITCH TSV, clean IDs, map descriptions, add metadata |
| 4 | Filter, Validate, Deduplicate & Export | Remove unmapped, validate, deduplicate, export |

## Final Output Schema

| Column | Description |
|--------|-------------|
| `head` | PubChem CID (chemical identifier) |
| `relation` | `ChemicalEntity_Gene` |
| `tail` | WormBase LocusTag (gene/protein identifier) |
| `head_type` | `ChemicalEntity` |
| `relation_type` | (NaN) |
| `tail_type` | `Gene` |
| `kg_source` | `STITCH` |
| `head_id_is` | `Pubchem` |
| `tail_id_is` | `WormBase_GeneSYMBOL` |
| `head_detail_name` | PubChem IUPAC chemical name |
| `tail_detail_name` | Gene description from NCBI |
| `species` | `C.elegans` |

## Data Download Instructions

All databases used in this notebook must be downloaded prior to execution.
Please refer to the **central download instructions document** for detailed steps:

📄 **[Download Instructions — Link to be added]**

### Required Files

| File | Source | Description |
|------|--------|-------------|
| `6239.protein_chemical.links.detailed.v5.0.tsv` | STITCH | Raw chemical–protein interaction file for C. elegans (tax_id: 6239) |
| `combined_df.pkl` | PubChem (pre-processed) | PubChem compound data with CID and IUPAC names |
| `Caenorhabditis_elegans.gene_info` | NCBI Gene | Gene annotations for C. elegans |

---
## Setup

Import required libraries and define all base paths.

In [2]:
import pandas as pd
import numpy as np

In [3]:
your_path_here = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

In [15]:
# =============================================================================
# BASE PATHS — Update these to match your local directory structure
# =============================================================================

# PubChem compound data (pre-processed pickle with CID, IUPAC names)
PUBCHEM_PATH = f'{your_path_here}databases_for_mapping/pubchem/combined_df.pkl'

# NCBI gene info for C. elegans
NCBI_GENE_INFO_PATH = f'{your_path_here}databases_for_mapping/ncbi/Caenorhabditis_elegans.gene_info'

# Raw STITCH chemical-protein interaction file for C. elegans
STITCH_RAW_PATH = f'{your_path_here}stitch/celegans/6239.protein_chemical.links.detailed.v5.0.tsv'

# Final output path
# OUTPUT_PATH = 'CHEMICALENTITY_GENE/ALL_CELE_CHEMICALENTITY_GENE.csv'
OUTPUT_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch/stitch_CELE_CHEMICALENTITY_GENE.csv'

---
## 1. Load PubChem (CID → IUPAC Name)

Load PubChem compound data to build a dictionary for annotating chemical entities with human-readable IUPAC names.

In [5]:
# =============================================================================
# Load PubChem compound data and build CID → IUPAC name dictionary
# =============================================================================

pubchem_df = pd.read_pickle(PUBCHEM_PATH)

# Dictionary: PubChem CID → IUPAC Name (for head_detail_name)
pubchem_cid_to_iupac = dict(zip(pubchem_df['PUBCHEM_COMPOUND_CID'], pubchem_df['PUBCHEM_IUPAC_NAME']))

print(f"PubChem compounds loaded: {len(pubchem_df):,}")
print(f"CID → IUPAC dictionary size: {len(pubchem_cid_to_iupac):,}")

# Free memory
del pubchem_df

PubChem compounds loaded: 119,177,440
CID → IUPAC dictionary size: 119,177,440


---
## 2. NCBI Gene Info — Build LocusTag → Description Dictionary

Load NCBI gene annotations for *C. elegans*. The `LocusTag` field has a `CELE_` prefix that needs to be stripped. In C. elegans, STITCH protein IDs correspond directly to WormBase LocusTags (e.g., `B0035.7`).

In [6]:
# =============================================================================
# Load NCBI gene info and build dictionaries
# =============================================================================

ncbi_cele_gene = pd.read_csv(NCBI_GENE_INFO_PATH, sep='\t')

# Strip 'CELE_' prefix from LocusTag
ncbi_cele_gene['LocusTag'] = ncbi_cele_gene['LocusTag'].str.replace('CELE_', '', regex=False)

# Dictionary: LocusTag → Symbol
ncbi_locustag_to_symbol = dict(zip(ncbi_cele_gene['LocusTag'], ncbi_cele_gene['Symbol']))

# Dictionary: LocusTag → Description (used for tail_detail_name)
ncbi_locustag_to_description = dict(zip(ncbi_cele_gene['LocusTag'], ncbi_cele_gene['description']))

print(f"NCBI genes loaded: {len(ncbi_cele_gene):,}")
print(f"LocusTag → Symbol dictionary size: {len(ncbi_locustag_to_symbol):,}")
print(f"LocusTag → Description dictionary size: {len(ncbi_locustag_to_description):,}")

NCBI genes loaded: 46,927
LocusTag → Symbol dictionary size: 46,927
LocusTag → Description dictionary size: 46,927


---
## 3. Load and Process STITCH Chemical–Protein Data

Load the raw STITCH file, clean chemical IDs (strip STITCH prefixes), strip species prefix from protein IDs, and annotate with descriptions.

**Note:** In C. elegans, STITCH protein IDs after stripping `6239.` are already WormBase LocusTags — no additional mapping (like gProfiler) is needed.

In [7]:
# =============================================================================
# Load raw STITCH chemical-protein interaction file
# =============================================================================

chem_gene_df = pd.read_csv(STITCH_RAW_PATH, sep='\t')
chem_gene_df = chem_gene_df.rename(columns={'chemical': 'head', 'protein': 'tail'})

# Strip species prefix '6239.' from protein IDs (reveals WormBase LocusTag)
chem_gene_df['tail'] = chem_gene_df['tail'].str.replace('6239.', '', regex=False)

print(f"Raw STITCH interactions loaded: {len(chem_gene_df):,}")

Raw STITCH interactions loaded: 25,946,333


In [8]:
# =============================================================================
# Clean chemical IDs and add KG metadata
# =============================================================================

def clean_chemical_id(chem_id):
    """Remove STITCH-specific prefixes and leading zeros from chemical IDs."""
    return chem_id.lstrip('CIDs').lstrip('CIDm').lstrip('0')

# Note: DataFrame.applymap() is deprecated in newer pandas — should use .map() instead
chem_gene_df[['head']] = chem_gene_df[['head']].applymap(clean_chemical_id)

# Add KG metadata
chem_gene_df['relation'] = 'ChemicalEntity_Gene'
chem_gene_df['head_type'] = 'ChemicalEntity'
chem_gene_df['tail_type'] = 'Gene'
chem_gene_df['relation_type'] = np.nan
chem_gene_df['kg_source'] = 'STITCH'

# Map descriptions
chem_gene_df['head_detail_name'] = chem_gene_df['head'].astype(str).map(pubchem_cid_to_iupac)
chem_gene_df['tail_detail_name'] = chem_gene_df['tail'].map(ncbi_locustag_to_description)

# Set identifier namespaces
chem_gene_df['head_id_is'] = 'Pubchem'
chem_gene_df['tail_id_is'] = 'WormBase_GeneSYMBOL'
chem_gene_df['species'] = 'C.elegans'

print(f"Chemical IDs cleaned, metadata added")
print(f"Head descriptions mapped: {chem_gene_df['head_detail_name'].notna().sum():,}")
print(f"Tail descriptions mapped: {chem_gene_df['tail_detail_name'].notna().sum():,}")

/tmp/ipykernel_1649344/4139177454.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chem_gene_df[['head']] = chem_gene_df[['head']].applymap(clean_chemical_id)


Chemical IDs cleaned, metadata added
Head descriptions mapped: 25,103,274
Tail descriptions mapped: 17,553,058


---
## 4. Filter, Validate, Deduplicate & Export

Filter out entries with unmapped descriptions, validate data quality, deduplicate, and export.

In [9]:
# =============================================================================
# Filter out entries with unmapped descriptions
# =============================================================================

before_count = len(chem_gene_df)
chem_gene_df = chem_gene_df[~chem_gene_df['head_detail_name'].isna()]
chem_gene_df = chem_gene_df[~chem_gene_df['tail_detail_name'].isna()]
after_count = len(chem_gene_df)

print(f"Before filtering: {before_count:,}")
print(f"After filtering: {after_count:,}")
print(f"Removed: {before_count - after_count:,} triples with unmapped entities")

Before filtering: 25,946,333
After filtering: 17,006,005
Removed: 8,940,328 triples with unmapped entities


In [10]:
# =============================================================================
# Data quality validation
# =============================================================================

true_nan_count = chem_gene_df.isna().sum()
string_nan_count = chem_gene_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})

print("NaN validation summary:")
nan_summary

NaN validation summary:


,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
tail,0,0,0
experimental,0,0,0
prediction,0,0,0
database,0,0,0
textmining,0,0,0
combined_score,0,0,0
relation,0,0,0
head_type,0,0,0
tail_type,0,0,0


In [11]:
# =============================================================================
# Deduplicate by grouping on (head, relation, tail)
# =============================================================================

group_cols = ['head', 'relation', 'tail']

def merge_sources(x):
    """Merge multiple kg_source values with '::' separator."""
    return '::'.join(sorted(set(x.dropna())))

chem_gene_dedup = chem_gene_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Triples before deduplication: {len(chem_gene_df):,}")
print(f"Triples after deduplication: {len(chem_gene_dedup):,}")
print(f"Duplicates removed: {len(chem_gene_df) - len(chem_gene_dedup):,}")
chem_gene_dedup.head()

Triples before deduplication: 17,006,005
Triples after deduplication: 9,845,825
Duplicates removed: 7,160,180


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,1,ChemicalEntity_Gene,B0024.12,ChemicalEntity,NaN,Gene,STITCH,Pubchem,WormBase_GeneSYMBOL,3-acetyloxy-4-(trimethylazaniumyl)butanoate,Glucosamine 6-phosphate N-acetyltransferase,C.elegans
1,1,ChemicalEntity_Gene,B0041.7,ChemicalEntity,NaN,Gene,STITCH,Pubchem,WormBase_GeneSYMBOL,3-acetyloxy-4-(trimethylazaniumyl)butanoate,Transcriptional regulator ATRX homolog,C.elegans
2,1,ChemicalEntity_Gene,B0272.4,ChemicalEntity,NaN,Gene,STITCH,Pubchem,WormBase_GeneSYMBOL,3-acetyloxy-4-(trimethylazaniumyl)butanoate,Uncharacterized protein,C.elegans
3,1,ChemicalEntity_Gene,B0365.1,ChemicalEntity,NaN,Gene,STITCH,Pubchem,WormBase_GeneSYMBOL,3-acetyloxy-4-(trimethylazaniumyl)butanoate,ATP-citrate synthase,C.elegans
4,1,ChemicalEntity_Gene,B0523.5,ChemicalEntity,NaN,Gene,STITCH,Pubchem,WormBase_GeneSYMBOL,3-acetyloxy-4-(trimethylazaniumyl)butanoate,Protein flightless-1 homolog,C.elegans


In [16]:
# chem_gene_dedup= pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/processed_data/

# # CHEMICALENTITY_GENE/ALL_CELE_CHEMICALENTITY_GENE.csv
OUTPUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch/stitch_CELE_CHEMICALENTITY_GENE.csv'

In [17]:
# =============================================================================
# Export final deduplicated Chemical-Gene KG triples
# =============================================================================

chem_gene_dedup.to_csv(OUTPUT_PATH, index=None)
print(f"Final output saved to: {OUTPUT_PATH}")
print(f"Total triples exported: {len(chem_gene_dedup):,}")

Final output saved to: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch/stitch_CELE_CHEMICALENTITY_GENE.csv
Total triples exported: 9,845,825


---
## Summary

This notebook processed STITCH Chemical–Protein interaction data for *C. elegans* into standardized Chemical–Gene KG triples:

1. **PubChem setup** — Loaded CID → IUPAC name dictionary
2. **NCBI annotation** — Loaded gene info, built LocusTag → Description dictionary
3. **STITCH processing** — Loaded raw interactions, cleaned chemical IDs, stripped species prefix, mapped descriptions
4. **Filtering** — Removed entries with unmapped chemicals or genes
5. **Deduplication** — Grouped by (head, relation, tail) for unique final triples

### Output

| File | Description |
|------|-------------|
| `/stitch_CELE_CHEMICALENTITY_GENE.csv` | Final: deduplicated relation-wise KG triples |